# Loading data from the folder

In [1]:
import zipfile
from pathlib import Path
import pandas as pd

GTFS_PATH = Path("./data/gtfs")

# 1. Collect all ZIP files recursively
zip_files = list(GTFS_PATH.rglob("*.zip"))
print(f"Found {len(zip_files)} GTFS zip files")

gtfs_data = {}

for zip_file in zip_files:
    # Build a unique key incorporating folder structure to prevent overwrites
    feed_key = "_".join(zip_file.relative_to(GTFS_PATH).parts)
    gtfs_data[feed_key] = {}

    with zipfile.ZipFile(zip_file) as z:
        # Standardize search for text files
        txt_files = [f for f in z.namelist() if f.lower().endswith(".txt")]
        
        for txt in txt_files:
            # Extract pure table name regardless of internal ZIP directory paths
            table_name = Path(txt).stem.lower()

            with z.open(txt) as file:
                # Force string dtype to protect leading zeros and prevent schema drift
                df = pd.read_csv(file, dtype=str, encoding="utf-8-sig", low_memory=False)
            
            gtfs_data[feed_key][table_name] = df

print(f"Successfully loaded feeds: {list(gtfs_data.keys())}")

# 2. Aggregation Helper Function
def aggregate_table(data_dict: dict, table_key: str) -> pd.DataFrame:
    frames = []
    for feed_id, tables in data_dict.items():
        if table_key in tables:
            temp = tables[table_key].copy()
            temp["feed_id"] = feed_id  # Compound source key tracking
            frames.append(temp)
    
    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

# Extract and combine tables deterministically
routes = aggregate_table(gtfs_data, "routes")
stops = aggregate_table(gtfs_data, "stops")
trips = aggregate_table(gtfs_data, "trips")
stop_times = aggregate_table(gtfs_data, "stop_times")
shapes = aggregate_table(gtfs_data, "shapes")

# 3. Robust Bus Route Filtering (Standard Code 3 & Extended Codes 700-716)
bus_codes = {"3"} | {str(i) for i in range(700, 716)}

if not routes.empty and "route_type" in routes.columns:
    bus_routes = routes[routes["route_type"].astype(str).str.strip().isin(bus_codes)].copy()
else:
    bus_routes = pd.DataFrame()

print(f"Total Routes: {len(routes)} | Bus Routes Identified: {len(bus_routes)}")

Found 8 GTFS zip files
Successfully loaded feeds: ['1_google_transit.zip', '10_google_transit.zip', '11_google_transit.zip', '2_google_transit.zip', '3_google_transit.zip', '4_google_transit.zip', '5_google_transit.zip', '6_google_transit.zip']
Total Routes: 1072 | Bus Routes Identified: 948


In [2]:
print("Bus Routes:", bus_routes.shape)
print("Stops:", stops.shape)
print("Trips:", trips.shape)
print("Stop Times:", stop_times.shape)
print("Shapes:", shapes.shape)


print("\n Bus Routes columns:")
print(bus_routes.columns.tolist())
print("\nStops columns:")
print(stops.columns.tolist())
print("\nStops times columns:")
print(stop_times.columns.tolist())
print("\nTrips columns:")
print(trips.columns.tolist())
print("\n Shapes columns:")
print(shapes.columns.tolist())

Bus Routes: (948, 8)
Stops: (31981, 12)
Trips: (314476, 10)
Stop Times: (11998317, 10)
Shapes: (13238055, 6)

 Bus Routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'feed_id']

Stops columns:
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'stop_url', 'location_type', 'parent_station', 'wheelchair_boarding', 'level_id', 'feed_id', 'stop_code', 'platform_code']

Stops times columns:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled', 'feed_id']

Trips columns:
['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'direction_id', 'block_id', 'wheelchair_accessible', 'bikes_allowed', 'feed_id']

 Shapes columns:
['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled', 'feed_id']


In [3]:
# dataset profiling
print("==================Routes===============")
print(bus_routes.head(3))
print("===============Stops==================")
print(stops.head(3))
print("================Stop times=================")
print(stop_times.head(3))
print("===============trips==================")
print(trips.head(3))
print("============== shapes==================")
print(shapes.head(3))


==================Routes===============
        route_id agency_id route_short_name                  route_long_name  \
14  11-box-vpt-1       NaN              NaN     Box Hill - Melbourne Airport   
15  11-SK4-vpt-1       NaN              NaN    Frankston - Melbourne Airport   
16  11-Ska-vpt-1       NaN              NaN  Melbourne City - Avalon Airport   

   route_type route_color route_text_color                feed_id  
14          3      ED1C24           FFFFFF  11_google_transit.zip  
15          3      ED1C24           FFFFFF  11_google_transit.zip  
16          3      ED1C24           FFFFFF  11_google_transit.zip  
===============Stops==================
  stop_id                stop_name      stop_lat      stop_lon  \
0   11212  Flinders Street Station  -37.81809481  144.96626579   
1   11213  Flinders Street Station  -37.81814377  144.96649165   
2   11214  Flinders Street Station  -37.81819840  144.96652423   

                                            stop_url location_t

## Adding bus route to other dataset to filter out bus routes only

In [4]:
bus_codes = {"3"} | {str(i) for i in range(700, 716)}
bus_routes = routes[
    routes["route_type"].astype(str)
          .str.strip()
          .isin(bus_codes)
].copy()

print(bus_routes.shape)
print(bus_routes.head(3))

(948, 8)
        route_id agency_id route_short_name                  route_long_name  \
14  11-box-vpt-1       NaN              NaN     Box Hill - Melbourne Airport   
15  11-SK4-vpt-1       NaN              NaN    Frankston - Melbourne Airport   
16  11-Ska-vpt-1       NaN              NaN  Melbourne City - Avalon Airport   

   route_type route_color route_text_color                feed_id  
14          3      ED1C24           FFFFFF  11_google_transit.zip  
15          3      ED1C24           FFFFFF  11_google_transit.zip  
16          3      ED1C24           FFFFFF  11_google_transit.zip  


In [5]:
# adding route_type into trips\
bus_trips = trips.merge(
    bus_routes[
        [
            "route_id",
            "route_type"
        ]
    ],
    on="route_id",
    how="inner"
)

print(bus_trips.shape)
print(bus_trips.head(3))
print(bus_trips["route_type"].unique())

(172456, 11)
       route_id service_id                trip_id          shape_id  \
0  11-box-vpt-1       T0_1  1.T0.11-box-vpt-1.1.H  11-box-vpt-1.1.H   
1  11-box-vpt-1       T2_1  1.T2.11-box-vpt-1.1.H  11-box-vpt-1.1.H   
2  11-box-vpt-1       T3_1  1.T3.11-box-vpt-1.1.H  11-box-vpt-1.1.H   

  trip_headsign direction_id block_id wheelchair_accessible bikes_allowed  \
0      Box Hill            0      NaN                     2           NaN   
1      Box Hill            0      NaN                     2           NaN   
2      Box Hill            0      NaN                     2           NaN   

                 feed_id route_type  
0  11_google_transit.zip          3  
1  11_google_transit.zip          3  
2  11_google_transit.zip          3  
<ArrowStringArray>
['3', '701']
Length: 2, dtype: str


In [6]:
# Adding route_type to stop_times to create bus_stop_times
bus_stop_times = stop_times.merge(
    bus_trips[
        [
            "trip_id",
            "route_id",
            "route_type",
            "shape_id"
        ]
    ],
    on="trip_id",
    how="inner"
)

print(bus_stop_times.shape)
print(bus_stop_times.head(3))
print(bus_stop_times["route_type"].unique())

(7026120, 13)
                 trip_id arrival_time departure_time stop_id stop_sequence  \
0  1.T0.11-box-vpt-1.1.H     20:15:00       20:15:00   48578             1   
1  1.T0.11-box-vpt-1.1.H     20:18:00       20:18:00   45511             2   
2  1.T0.11-box-vpt-1.1.H     20:43:00       20:43:00   20668             3   

  stop_headsign pickup_type drop_off_type shape_dist_traveled  \
0           NaN           0             0                0.00   
1           NaN           0             1              397.60   
2           NaN           0             0            26921.84   

                 feed_id      route_id route_type          shape_id  
0  11_google_transit.zip  11-box-vpt-1          3  11-box-vpt-1.1.H  
1  11_google_transit.zip  11-box-vpt-1          3  11-box-vpt-1.1.H  
2  11_google_transit.zip  11-box-vpt-1          3  11-box-vpt-1.1.H  
<ArrowStringArray>
['3', '701']
Length: 2, dtype: str


In [7]:
# use the filtered stop_id to create bus_stops dataset from stops dataset
bus_stops = stops.merge(
    bus_stop_times[
        [
            "stop_id"
        ]
    ].drop_duplicates(),
    on="stop_id",
    how="inner"
)

print(bus_stops.shape)
print(bus_stops.head(3))

(26270, 12)
  stop_id                stop_name      stop_lat      stop_lon  \
0    1100  Broadway/Glen Huntly Rd  -37.88211832  144.98222007   
1    1109    Inkerman St/Barkly St  -37.86372960  144.98193743   
2    1414            Bay St/New St  -37.90409768  144.99327749   

                                            stop_url location_type  \
0  https://transport.vic.gov.au/stop/12195/?utm_s...           NaN   
1  https://transport.vic.gov.au/stop/12262/?utm_s...           NaN   
2  https://transport.vic.gov.au/stop/15099/?utm_s...           NaN   

  parent_station wheelchair_boarding level_id                feed_id  \
0            NaN                   0  Level 0  11_google_transit.zip   
1            NaN                   0  Level 0  11_google_transit.zip   
2            NaN                   0  Level 0  11_google_transit.zip   

  stop_code platform_code  
0      1100           NaN  
1      1109           NaN  
2      1414           NaN  


In [8]:
# Bus shapes created from bus_trips dataset
bus_shapes = shapes.merge(
    bus_trips[
        [
            "shape_id",
            "route_id",
            "route_type"
        ]
    ].drop_duplicates(),
    on="shape_id",
    how="inner"
)

print(bus_shapes.shape)
print(bus_shapes.head(3))

(1652751, 8)
           shape_id  shape_pt_lat  shape_pt_lon shape_pt_sequence  \
0  11-box-vpt-1.1.H  -37.67231992  144.84921649                 1   
1  11-box-vpt-1.1.H  -37.67206057  144.84916774                 2   
2  11-box-vpt-1.1.H  -37.67200528  144.84916038                 3   

  shape_dist_traveled                feed_id      route_id route_type  
0                0.00  11_google_transit.zip  11-box-vpt-1          3  
1               29.16  11_google_transit.zip  11-box-vpt-1          3  
2               35.34  11_google_transit.zip  11-box-vpt-1          3  


# Save datset   

In [9]:
# 1. Define output directory and ensure existence
PROCESSED_DATA_PATH = Path("./processed_data")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# 2. Map all transformed GTFS bus datasets to their target file names
gtfs_bus_datasets = {
    "bus_routes": (bus_routes, "bus_routes.csv"),
    "bus_trips": (bus_trips, "bus_trips.csv"),
    "bus_stop_times": (bus_stop_times, "bus_stop_times.csv"),
    "bus_stops": (bus_stops, "bus_stops.csv"),
    "bus_shapes": (bus_shapes, "bus_shapes.csv")
}

# 3. Export datasets to disk and print structured inspection outputs
for name, (df, filename) in gtfs_bus_datasets.items():
    file_path = PROCESSED_DATA_PATH / filename
    
    # Save to CSV without writing index column
    df.to_csv(file_path, index=False)
    
    # Print formatted metadata matching standard signature
    print("=" * 60)
    print(f" DATASET: {name} (Saved to: {file_path})")
    print("=" * 60)
    print(df.columns.to_list())
    print("\n--- Head (3) ---")
    print(df.head(3))
    print("\n--- Info ---")
    df.info()
    print("\n" + "\n")

 DATASET: bus_routes (Saved to: processed_data\bus_routes.csv)
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'feed_id']

--- Head (3) ---
        route_id agency_id route_short_name                  route_long_name  \
14  11-box-vpt-1       NaN              NaN     Box Hill - Melbourne Airport   
15  11-SK4-vpt-1       NaN              NaN    Frankston - Melbourne Airport   
16  11-Ska-vpt-1       NaN              NaN  Melbourne City - Avalon Airport   

   route_type route_color route_text_color                feed_id  
14          3      ED1C24           FFFFFF  11_google_transit.zip  
15          3      ED1C24           FFFFFF  11_google_transit.zip  
16          3      ED1C24           FFFFFF  11_google_transit.zip  

--- Info ---
<class 'pandas.DataFrame'>
Index: 948 entries, 14 to 1071
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ro